# "THE PRICE IS RIGHT" 顶点项目

本周——基于抓取的 Amazon 数据，构建一个能根据描述预测某物价格的模型

# 日程安排

DAY 1：数据整理（Data Curation）  
DAY 2：数据预处理（Data Pre-processing）  
DAY 3：评估、基线、传统机器学习  
DAY 4：深度学习与 LLM  
DAY 5：微调前沿模型  

## DAY 3：评估、基线、传统机器学习

今天我们将编写一些简单模型来预测产品价格

我们将使用一种方法来评估模型性能

并使用传统机器学习测试一些基线模型

In [ ]:
# 导入：sklearn 线性回归 / 随机森林 / 词袋；evaluate 统一算定价误差

import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate
from pricer.items import Item

In [ ]:
# LITE_MODE：True 用小数据快速实验

LITE_MODE = False

In [ ]:
# 从 Hub 加载 train / val / test

username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
# 基线 1：完全随机报价（1–999）——预期很差，用来定下限

def random_pricer(item):
    return random.randrange(1,1000)

In [ ]:
# 评估随机报价器在测试集上的误差

random.seed(42)
evaluate(random_pricer, test)

In [ ]:
# 挺有趣的！
# 我们可以做得更好——这里是另一个相当简单的模型

training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

def constant_pricer(item):
    return training_average

In [ ]:
# 评估「恒定均值」报价器

evaluate(constant_pricer, test)

In [ ]:
# 手工特征：重量、是否缺重量、摘要文本长度

def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

In [ ]:
# 把 item 列表转成带特征列与 price 标签的 DataFrame

def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

In [ ]:
# 传统线性回归！

np.random.seed(42)

# 分离特征与目标
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# 训练线性回归
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

# 预测测试集并评估
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

In [ ]:
# 用训练好的线性回归对单个 item 预测价格

def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

In [ ]:
# 评估线性回归报价器

evaluate(linear_regression_pricer, test)

In [ ]:
# 准备 NLP 特征：价格标签 + 商品 summary 文本

prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
# 词袋模型（Bag-of-Words）：CountVectorizer 把文本变成词频向量
# max_features 限制词典大小；stop_words 去掉 the/a 等停用词

np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)


In [ ]:
# 以下是它选出的 1,000 个最常见词，不包括 “停用词”：

selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

In [ ]:
# 在词袋特征上训练线性回归（NLP + LR）

regressor = LinearRegression()
regressor.fit(X, prices)

In [ ]:
# 预测时同样先 vectorize summary；价格截断为非负

def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

In [ ]:
# 评估 NLP + 线性回归

evaluate(natural_language_linear_regression_pricer, test)

In [ ]:
# 随机森林：非线性集成模型；子集 15_000 条以控制训练时间

subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

## 随机森林（Random Forest）模型

随机森林是一种 “**集成（ensemble）**” 算法，意味着它组合许多更小的算法来做出更好的预测。

它使用一种非常简单的机器学习算法，叫做 **决策树（decision tree）**。决策树通过检查输入中特征的值来做预测。就像带有 IF 语句的流程图。决策树非常快速、简单，但容易过拟合。

在我们的例子中，“特征”是向量的各个元素——换句话说，是某个词在产品描述中出现的次数。

所以你可以这样理解：

**决策树**  
\- 如果词 “TV” 出现超过 3 次，那么  
-- 如果词 “LED” 出现超过 2 次，那么  
--- 如果词 “HD” 至少出现一次，那么  
---- 价格 = $500


使用随机森林时，会创建多棵决策树。每一棵用不同的随机数据子集、以及不同的随机特征子集来训练。你可以看到上面我们指定了 100 棵树，这是默认值。

然后随机森林模型简单地对所有树取平均，得到最终结果。

In [ ]:
# 随机森林推理封装

def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [ ]:
# 评估随机森林

evaluate(random_forest, test)

In [ ]:
# 如果你想保存模型可以这样做，尤其是在更大的数据集上运行时

# import joblib
# joblib.dump(rf_model, "random_forest.joblib")

## 介绍 XGBoost

与随机森林一样，XGBoost 也是一种组合多棵决策树的集成模型。

但与随机森林不同，XGBoost 一棵接一棵地构建树，每一棵后续的树用「梯度下降」来纠正先前树的误差。

它比随机森林快得多，因此我们可以在完整数据集上运行，并且通常在泛化上表现更好。

**如果这个导入不工作，请跳过！它不是必需的。在 Mac 上，你可能需要在终端执行 `brew install libomp`。**

In [ ]:
# 导入 XGBoost：梯度提升树，常比随机森林更强

import xgboost as xgb

In [ ]:
# 训练 XGBoost 回归器（树的数量较多，可能需一点时间）

np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

In [ ]:
# XGBoost 推理封装

def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [ ]:
# 评估 XGBoost

evaluate(xg_boost, test)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">传统机器学习不只是用来了解历史；它今天在工业界仍被大量使用，尤其是特征清晰可辨的任务。值得花时间探索这些算法并在此实验。看看你能否用传统机器学习打败我的数字！我在全部 800,000 条训练数据上运行了随机森林。大约花了 15 小时，最终得到低至 $56.40 的误差。传统机器学习可以做得很好——自己试试吧。</span>
        </td>
    </tr>
</table>